# M5 - Supply Chain Dataset Cleaning

This notebook cleans the project's `DataCoSupplyChainDataset.csv` without modifying the raw file. It removes surrounding whitespace from text fields, standardizes the two documented timestamp columns, validates the output, and saves the cleaned CSV in `data/interim/`.

No rows or columns are dropped, no missing values are imputed, and status labels are not recoded. This preserves order-item records and keeps fields available for later analysis.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASET_NAME = 'DataCoSupplyChainDataset.csv'
ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / 'data' / 'raw' / DATASET_NAME).is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        f'Could not locate data/raw/{DATASET_NAME} from {Path.cwd()} or its parent directories.'
    )

RAW_PATH = ROOT / 'data' / 'raw' / DATASET_NAME
INTERIM_DIR = ROOT / 'data' / 'interim'
REPORTS_DIR = ROOT / 'reports'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH, encoding='latin1', low_memory=False)
print(f'Project root: {ROOT}')
print(f'Raw dataset: {RAW_PATH}')
print(f'Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')
display(df_raw.head())

Project root: q:\Family Data\Data analyst\ecommerce-logistics-analysis
Raw dataset: q:\Family Data\Data analyst\ecommerce-logistics-analysis\data\raw\DataCoSupplyChainDataset.csv
Raw shape: 180,519 rows x 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## Inspect the raw data

Review missingness and duplicate counts before cleaning. The project's data dictionary notes that `Product Description` is entirely missing and `Order Zipcode` is heavily incomplete; both columns are retained without imputation so downstream decisions remain explicit.

In [2]:
raw_missing = df_raw.isna().sum().rename('Missing values').to_frame()
raw_missing['Missing percent'] = (raw_missing['Missing values'] / len(df_raw) * 100).round(2)
raw_missing = raw_missing[raw_missing['Missing values'] > 0].sort_values(
    'Missing values', ascending=False
)

print(f'Exact duplicate rows: {df_raw.duplicated().sum():,}')
display(raw_missing)
display(df_raw.dtypes.astype(str).value_counts().rename_axis('Data type').to_frame('Column count'))

Exact duplicate rows: 0


,Missing values,Missing percent
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


,Column count
Data type,
object,24
float64,15
int64,14


## Cleaning rules

- Strip leading and trailing whitespace from all text columns while preserving missing values.
- Parse `order date (DateOrders)` and `shipping date (DateOrders)` and format valid timestamps as `YYYY-MM-DD HH:MM:SS`.
- Stop with an explicit error if a non-missing timestamp cannot be parsed; do not silently convert bad values to missing data.
- Preserve the original row count, columns, identifiers, missing values, and business labels. Repeated `Order Id` values are expected at the order-item grain.

In [3]:
df_clean = df_raw.copy()
text_columns = df_clean.select_dtypes(include=['object', 'string']).columns.tolist()
whitespace_trimmed = {}

for column in text_columns:
    non_missing = df_clean[column].notna()
    original = df_clean[column]
    trimmed = original.str.strip()
    whitespace_trimmed[column] = int((non_missing & original.ne(trimmed)).sum())
    df_clean[column] = trimmed

date_columns = ['order date (DateOrders)', 'shipping date (DateOrders)']
missing_date_columns = [column for column in date_columns if column not in df_clean.columns]
if missing_date_columns:
    raise KeyError(f'Required timestamp columns are missing: {missing_date_columns}')

unparsed_dates = {}
for column in date_columns:
    original = df_clean[column]
    parsed = pd.to_datetime(original, errors='coerce')
    invalid = original.notna() & parsed.isna()
    unparsed_dates[column] = int(invalid.sum())
    if invalid.any():
        examples = original.loc[invalid].head(5).tolist()
        raise ValueError(f'Could not parse {column!r}; examples: {examples}')
    valid = parsed.notna()
    df_clean.loc[valid, column] = parsed.loc[valid].dt.strftime('%Y-%m-%d %H:%M:%S')

assert df_clean.shape == df_raw.shape, 'Cleaning changed the dataset shape.'
assert df_clean.columns.equals(df_raw.columns), 'Cleaning changed the dataset columns.'
print(f'Text columns checked: {len(text_columns)}')
print(f'Rows and columns preserved: {df_clean.shape}')

Text columns checked: 24
Rows and columns preserved: (180519, 53)


## Validate and save

The quality summary records row/column counts, duplicate counts, and each column's missingness before and after cleaning. It is written separately from the cleaned dataset.

In [4]:
quality_summary = pd.DataFrame(
    [
        {
            'Metric': 'Rows',
            'Before cleaning': len(df_raw),
            'After cleaning': len(df_clean),
        },
        {
            'Metric': 'Columns',
            'Before cleaning': len(df_raw.columns),
            'After cleaning': len(df_clean.columns),
        },
        {
            'Metric': 'Exact duplicate rows',
            'Before cleaning': int(df_raw.duplicated().sum()),
            'After cleaning': int(df_clean.duplicated().sum()),
        },
        {
            'Metric': 'Total missing values',
            'Before cleaning': int(df_raw.isna().sum().sum()),
            'After cleaning': int(df_clean.isna().sum().sum()),
        },
    ]
)

missing_summary = pd.DataFrame(
    {
        'Metric': [f'Missing values: {column}' for column in df_raw.columns],
        'Before cleaning': df_raw.isna().sum().to_numpy(),
        'After cleaning': df_clean.isna().sum().to_numpy(),
    }
)
cleaning_summary = pd.concat([quality_summary, missing_summary], ignore_index=True)

whitespace_summary = pd.DataFrame(
    [
        {
            'Column': column,
            'Values trimmed': count,
            'Remaining surrounding whitespace': int(
                (df_clean[column].notna() & df_clean[column].ne(df_clean[column].str.strip())).sum()
            ),
        }
        for column, count in whitespace_trimmed.items()
    ]
)
date_summary = pd.DataFrame(
    [{'Column': column, 'Unparseable non-missing values': count} for column, count in unparsed_dates.items()]
)

assert len(df_clean) == len(df_raw)
assert df_clean.columns.equals(df_raw.columns)
assert df_clean.isna().sum().equals(df_raw.isna().sum()), 'Cleaning unexpectedly changed missing values.'
assert all(count == 0 for count in unparsed_dates.values())
assert whitespace_summary['Remaining surrounding whitespace'].eq(0).all()

display(cleaning_summary.head(8))
display(whitespace_summary)
display(date_summary)

,Metric,Before cleaning,After cleaning
0,Rows,180519,180519
1,Columns,53,53
2,Exact duplicate rows,0,0
3,Total missing values,336209,336209
4,Missing values: Type,0,0
5,Missing values: Days for shipping (real),0,0
6,Missing values: Days for shipment (scheduled),0,0
7,Missing values: Benefit per order,0,0


,Column,Values trimmed,Remaining surrounding whitespace
0,Type,0,0
1,Delivery Status,0,0
2,Category Name,1475,0
3,Customer City,0,0
4,Customer Country,0,0
5,Customer Email,0,0
6,Customer Fname,0,0
7,Customer Lname,0,0
8,Customer Password,0,0
9,Customer Segment,0,0


,Column,Unparseable non-missing values
0,order date (DateOrders),0
1,shipping date (DateOrders),0


In [5]:
CLEANED_PATH = INTERIM_DIR / 'DataCoSupplyChain_M5_cleaned.csv'
QUALITY_PATH = REPORTS_DIR / 'M5_cleaning_quality_summary.csv'

df_clean.to_csv(CLEANED_PATH, index=False, encoding='utf-8')
cleaning_summary.to_csv(QUALITY_PATH, index=False, encoding='utf-8')

if not CLEANED_PATH.is_file() or CLEANED_PATH.stat().st_size == 0:
    raise OSError(f'Cleaned dataset was not written successfully: {CLEANED_PATH}')

print(f'Cleaned dataset saved to: {CLEANED_PATH}')
print(f'Quality summary saved to: {QUALITY_PATH}')
print(f'Cleaned shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]:,} columns')

Cleaned dataset saved to: q:\Family Data\Data analyst\ecommerce-logistics-analysis\data\interim\DataCoSupplyChain_M5_cleaned.csv
Quality summary saved to: q:\Family Data\Data analyst\ecommerce-logistics-analysis\reports\M5_cleaning_quality_summary.csv
Cleaned shape: 180,519 rows x 53 columns


In [6]:
df_saved = pd.read_csv(CLEANED_PATH, encoding='utf-8', low_memory=False)
assert df_saved.shape == df_raw.shape, 'Saved CSV does not have the expected shape.'
assert df_saved.columns.equals(df_raw.columns), 'Saved CSV columns differ from the source.'
print('Round-trip validation passed.')
print(f'Reloaded shape: {df_saved.shape[0]:,} rows x {df_saved.shape[1]:,} columns')

Round-trip validation passed.
Reloaded shape: 180,519 rows x 53 columns


## Handoff

Use the cleaned CSV in `data/interim/` for subsequent project analysis. Review the quality summary and the data dictionary before deciding whether to exclude high-missingness or personal-data fields. The raw source remains unchanged.